In [8]:
label_desc = {
            0:  "person",
            1:  "ear",
            2:  "ear-mufs",
            3:  "face",
            4:  "face-guard",
            5:  "face-mask",
            6:  "foot",
            7:  "tool",
            8:  "glasses",
            9:  "gloves",
            10: "helmet",
            11: "hands",
            12: "head",
            13: "medical-suit",
            14: "shoes",
            15: "safety-suit",
            16: "safety-vest"
            }

In [6]:
def get_dict_key(label_dict : dict, label_name:str):
    try:
        for key, value in label_dict.items():
            if value == label_name:
                return key
    except:
        # print('There is no value for that label')
        return None

In [18]:
object_test = "ear"

index = get_dict_key(label_desc, object_test)
if index:
    print(f"Object ada di index = {index}")
else:
    print(f"Label {object_test} gak ada bro")

Object ada di index = 1


## Mengecek ada label apa di dalam file-file .xml nyah

In [ ]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET

def list_file_in_folder(path:str|Path):
    files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
    return files

def scrap_xml (xml_path):
    labels = []
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall('object'):
        name = obj.find('name').text
        labels.append(name)
    return labels

xml_location = "voc_labels"

files = list_file_in_folder(xml_location)

all_label = []

for file in files:
    # file = files[i]
    file_path = f"{xml_location}/{file}"
    labels_found = scrap_xml(file_path)
    all_label.extend(labels_found)

all_label = set(all_label)
all_label = list(all_label)

for llabel in all_label:
    print(f"{llabel}")

face
shoes
medical-suit
safety-suit
ear
ear-mufs
gloves
foot
face-mask-medical
safety-vest
glasses
tools
face-guard
helmet
head
person
hands


# Unpacking data from xml

In [4]:
def normalize_bbox (x_min:int,y_min:int,x_max:int,y_max:int, image_width:int, image_height:int):
    bbox_width = x_max - x_min
    bbox_height = y_max - y_min
    x_center = (bbox_width/2) + x_min
    y_center = (bbox_height/2) + y_min

    xcent_norm = x_center / (image_width-1)
    ycent_norm = y_center / (image_height-1)    
    imwidth_norm = bbox_width / (image_width-1)
    imheight_norm = bbox_height / (image_height-1)
    
    #<x_center> <y_center> <width> <height>
    return round(xcent_norm,6), round(ycent_norm,6), round(imwidth_norm,6), round(imheight_norm,6)

# How to use the function

# xcent_norm, ycent_norm, imwidth_norm, imheight_norm = normalize_bbox(x_min=627,
#                                                         y_min=546,
#                                                         x_max=1902,
#                                                         y_max=2063,
#                                                         image_width=5760,
#                                                         image_height=3840
#                                                     )
# print(f'norm_value = {xcent_norm}, {ycent_norm}, {imwidth_norm}, {imheight_norm}')

In [ ]:
from pathlib import Path
import xml.etree.cElementTree as ET

xml_path = Path('/Users/mamuflih/Documents/01_coding/voc2yolo-converter/dataset/voc_labels/building-construction-building-site-constructing.xml')

def extract_yolo_from_voc(xml_path: int|Path, ):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Untuk 'block' size
    file_name = root.find('filename')

    size = root.find('size')
    width = int(size.find('width').text)
    height = int(size.find('height').text)

    # idk what depth will do in all of these things
    # depth = int(size.find('depth').text)

    lines = []

    # Untuk 'block' object dalam satu xml ada banyak object maka menggunakan findall
    for object in root.findall('object'):
        class_name = object.find('name').text
        # Ubah nama object menjadio id object
        class_id = get_dict_key(
                label_dict=label_desc,
                label_name=class_name
                )
        
        # Kalau nanti gw bisa melihat variasi dari label xml untuk model computer vision lain, mungkin ini bisa digunakan dan dijadikan filter, untuk sekarang di comment dulu
        # object_pose = object.find('pose')
        # object_truncated = int(object.find('truncated'))
        # object_difficult = int(object.find('difficult'))

        # Untuk block 'bndbox'
        bbox = object.find('bndbox')

        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        xcent_norm, ycent_norm, w_norm, h_norm = normalize_bbox(
                                                    x_min=xmin,
                                                    y_min=ymin,
                                                    x_max=xmax,
                                                    y_max=ymax,
                                                    image_width=width,
                                                    image_height=height
                                                ) 
        
        line = f'{class_id} {xcent_norm} {ycent_norm} {w_norm} {h_norm}'
        lines.append(line)
        return {
                'file_name':file_name,
                'lines':lines,
            }
    
def verify_label_image(image_path:str|Path, image_folder_path:str|Path):
    # buat pengecekan string atau Path (jadikan path full lalu jadikan string, ambil nama filenya saja)
    all_image_path = list_file_in_folder(path=image_folder_path)
    if image_path in all_image_path:
        return True
    else:
        return False

def convert_yolo_from_voc(txt_lines: list, output_path: int|Path, image_path: int|Path, images_folder: int|Path, save_invalid: bool = False):
    image_in_path = verify_label_image(image_path=image_path, image_folder_path=images_folder)
    
    if not image_in_path:
        print("Image for this label isn't available")
        if save_invalid:
            print("Label not saved because image for this label isn't available")
            return  False
    
    # Simpan label jika image ada atau save_invalid=False
    with open(output_path, "w") as f:
        for line in txt_lines:
            f.write(line + "\n")
    print(f"Label saved in {output_path}")
    
    return True

            
    

In [11]:
import xml.etree.cElementTree as ET
from pathlib import Path

xml_path = Path('/Users/mamuflih/Documents/01_coding/voc2yolo-converter/dataset/voc_labels/building-construction-building-site-constructing.xml')

tree = ET.parse(xml_path)
root = tree.getroot()

# Untuk 'block' size
file_name = root.find('filename')

size = root.find('size')
width = int(size.find('width').text)
height = int(size.find('height').text)
depth = int(size.find('depth').text)

lines = []

# Untuk 'block' object dalam satu xml ada banyak object maka menggunakan findall
for object in root.findall('object'):
    class_name = object.find('name').text
    # Ubah nama object menjadio id object
    class_id = get_dict_key(
            label_dict=label_desc,
            label_name=class_name
            )
    
    # Kalau nanti gw bisa melihat variasi dari label xml untuk model computer vision lain, mungkin ini bisa digunakan dan dijadikan filter, untuk sekarang di comment dulu
    # object_pose = object.find('pose')
    # object_truncated = int(object.find('truncated'))
    # object_difficult = int(object.find('difficult'))

    # Untuk block 'bndbox'
    bbox = object.find('bndbox')

    xmin = int(bbox.find('xmin').text)
    ymin = int(bbox.find('ymin').text)
    xmax = int(bbox.find('xmax').text)
    ymax = int(bbox.find('ymax').text)

    xcent_norm, ycent_norm, w_norm, h_norm = normalize_bbox(
                                                x_min=xmin,
                                                y_min=ymin,
                                                x_max=xmax,
                                                y_max=ymax,
                                                image_width=width,
                                                image_height=height
                                            ) 
    
    line = f'{class_id} {xcent_norm} {ycent_norm} {w_norm} {h_norm}'
    # lines.append(line)
    print(line)

9 0.219569 0.339802 0.221393 0.395155


In [33]:
xml_location = "voc_labels"

files = list_file_in_folder(xml_location)

for i in range (1):
    file = files[i]
    file_path = f"{xml_location}/{file}"
    image_filename, list_label = scrap_xml(file_path, label_desc)
    print (image_filename)
    print (list_label)

pexels-photo-6205737.jpeg
['11 1149 1170 3072 2842', '11 3236 75 4364 1416', '0 0 0 4367 2837']


In [12]:
xml_location = "voc_labels"

files = list_file_in_folder(xml_location)

for i in range (1):
    file = files[i]
    file_path = f"{xml_location}/{file}"
    scrap_xml(file_path)

Filename: pexels-photo-6205737.jpeg
Image size: 5616 x 3744 Depth: 3

Object: hands
Pose: Unspecified, Difficult: 0
BBox: (1149, 1170) to (3072, 2842)

Object: hands
Pose: Unspecified, Difficult: 0
BBox: (3236, 75) to (4364, 1416)

Object: person
Pose: Unspecified, Difficult: 0
BBox: (0, 0) to (4367, 2837)
